# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`
This notebook demonstrates how to explore and process the FAIR² dataset using the `mlcroissant` library, referencing all entities with their Croissant `@id` values as specified by the schema.

### Dataset Source
The source Croissant schema JSON-LD:
```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load metadata and dataset
dataset = mlc.Dataset(croissant_url)

# Display dataset high-level metadata
md = dataset.metadata
print(f"Dataset Name: {md.name}")
print(f"Version: {md.version}")
print(f"Identifier: {md.identifier}")
print(f"Description: {md.description}")


## 2. Data Overview
Review available record sets, fields, and their `@id`s.

The primary way to reference entities in Croissant is their `@id`. Let's list all record sets, their fields, and available columns. 

In [ ]:
# List all record sets with their @id

record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record set(s)")
for rs in record_sets:
    print(f"\nRecordSet: {rs['@id']}")
    print(f"  name: {rs.get('name', None)}")
    print(f"  description: {rs.get('description', None)}")
    # Fields (@id for each field)
    if 'field' in rs:
        print("  Fields:")
        for f in rs['field']:
            print(f"   - {f['@id']} (name: {f.get('name',None)}, dataType: {f.get('dataType',None)})")
    # Columns (for tabular sources)
    if 'column' in rs:
        print("  Columns:")
        for c in rs['column']:
            print(f"   - {c['@id']} (name: {c.get('name',None)})")


## 3. Data Extraction
Let's select a record set and load its records as a pandas DataFrame for further analysis. All lookups and columns will be referenced by their Croissant `@id`s.

Below, we demonstrate extracting all available record sets. Substitute the `@id` with the actual record set you want to analyze if needed.

In [ ]:
# Extract all record sets by their @id
dataframes = {}
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
print(f"Extracting these record sets by @id: {record_set_ids}")

for rs_id in record_set_ids:
    print(f"\nReading records from record set: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded DataFrame shape: {df.shape}")
    if not df.empty:
        print("Columns (@id): ", list(df.columns))
        display(df.head(3))


## 4. Exploratory Data Analysis (EDA)
We will select a numeric field using its `@id`, filter out records based on a threshold, normalize it, and group by a categorical field (all referenced by their `@id`).

Replace the field `@id` values below with those from the overview if your record set has different field names.

In [ ]:
# Pick the primary record set (assuming the first one is the main clinical table)
main_rs_id = record_set_ids[0]
main_df = dataframes[main_rs_id].copy()

# List numeric-type field @id candidates to choose one for demonstration
if not main_df.empty:
    print("Available columns (@id):", list(main_df.columns))

    # Example: Let's pick a field likely to be numeric, e.g. 'Age' (check actual @id from the overview cell)
    # For demonstration, substitute with actual @id ('cr:field:age' or similar) if present
    # If not present, pick any numeric column
    numeric_field_id = None
    for col in main_df.columns:
        if 'age' in col.lower() or 'interval' in col.lower() or main_df[col].dtype in [np.int64, np.float64]:
            numeric_field_id = col
            break
    if not numeric_field_id:
        # Default to the first numeric column
        for col in main_df.select_dtypes(include=[np.number]).columns:
            numeric_field_id = col
            break

    if numeric_field_id:
        print(f"Numeric field chosen for EDA: {numeric_field_id}")
        # Demonstrate filtering
        threshold = main_df[numeric_field_id].mean() if main_df[numeric_field_id].dtype in [np.float64,np.int64] else 0
        filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold}")
        display(filtered_df.head(3))

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Pick a group field (categorical, e.g. 'Sex' or 'cr:field:sex', or any 'site' or status info field)
        group_field_id = None
        for col in main_df.columns:
            # Heuristics: pick field likely to be categorical
            if any(key in col.lower() for key in ['site','sex','status','msi','group','anatomical']):
                group_field_id = col
                break
        # If no obvious group, search for a column of dtype 'object' with a small number (<=10) of unique values
        if not group_field_id:
            for col in main_df.select_dtypes(include='object').columns:
                if main_df[col].nunique() <= 10:
                    group_field_id = col
                    break
        if group_field_id:
            print(f"Grouping by: {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            display(grouped_df.head())
        else:
            print("No suitable group field found for grouping.")
    else:
        print("No numeric field found for EDA.")

## 5. Visualization
Let's visualize the distribution of the selected numeric field, and—if a group field is available—show boxplots grouped by the categorical variable, referencing each field by `@id`.

In [ ]:
# Visualization of numeric field's distribution and grouped boxplot if group field exists
if not main_df.empty and numeric_field_id:
    plt.figure(figsize=(8,4))
    main_df[numeric_field_id].hist(bins=20, edgecolor='black')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    if group_field_id:
        plt.figure(figsize=(10,5))
        main_df.boxplot(column=numeric_field_id, by=group_field_id, rot=45)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.suptitle("")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
In this notebook, we've demonstrated how to load and explore the FAIR² clinical dataset using the `mlcroissant` library, while referencing dataset entities by their Croissant `@id` throughout. We performed EDA on numerical and categorical fields, filtered and normalized data, and created visualizations to aid understanding for downstream analysis.

You can adapt this notebook further to analyze additional fields or record sets using their `@id` as described in the Croissant schema.